# Real-Time Financial Fraud Detection & Investigation: Exploratory Data Analysis (EDA)
**Author:** Senior Machine Learning Engineer & Data Scientist  
**Dataset:** ULB Credit Card Fraud Detection (Kaggle)  
**Objective:** Uncover class imbalance dynamics, evaluate feature separability (V1-V28 PCA projections), audit transaction amount distributions, and prepare leak-free preprocessing pipelines.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set project root
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data.load_data import load_dataset
from src.data.validate_data import DataValidator

print("Environment initialized successfully.")

## 1. Dataset Loading and Schema Validation
Loads either the raw `creditcard.csv` or synthetic fallback data for demonstration.

In [ ]:
df, is_demo = load_dataset(use_demo_if_missing=True)
print(f"Loaded dataset shape: {df.shape} (Demo mode: {is_demo})")
df.head()

## 2. Data Quality & Imbalance Audit
Evaluate missingness, duplicate records, and target distribution.

In [ ]:
validator = DataValidator()
quality_report = validator.generate_quality_report(df, is_demo=is_demo)

print("Total rows:", quality_report["total_rows"])
print("Total missing:", quality_report["missing_values"]["total_missing"])
print("Duplicate records:", quality_report["duplicate_records"])
print("Target stats:", quality_report["target_distribution"])

## 3. Class Imbalance Visualization
Demonstrating why naive accuracy is catastrophically misleading in fraud detection.

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="Class", palette=["#10b981", "#ef4444"])
plt.title("Class Distribution: Legitimate (0) vs Fraudulent (1)")
plt.xlabel("Transaction Class")
plt.ylabel("Count")
plt.yscale("log")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## 4. Transaction Amount & Time Distributions
Financial transaction amounts exhibit extreme positive skewness and long right tails, requiring robust scaling.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Amount distribution
sns.histplot(df["Amount"], bins=50, ax=axes[0], kde=True, color="#38bdf8")
axes[0].set_title("Distribution of Transaction Amounts")
axes[0].set_xlabel("Amount ($)")
axes[0].set_yscale("log")

# Time distribution
sns.histplot(df["Time"], bins=50, ax=axes[1], kde=True, color="#818cf8")
axes[1].set_title("Distribution of Transaction Times (Seconds)")
axes[1].set_xlabel("Elapsed Time (s)")

plt.tight_layout()
plt.show()

## 5. Feature Separability: Top Fraud Discriminators
Analysis of prominent PCA features (V14, V17, V12, V4) showing distinct distributional shifts between fraud and normal transactions.

In [ ]:
key_features = ["V14", "V17", "V12", "V4"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    sns.boxplot(x="Class", y=feat, data=df, ax=axes[i], palette=["#10b981", "#ef4444"])
    axes[i].set_title(f"{feat} vs Fraud Class")
    axes[i].grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Correlation Heatmap
Verifying correlation among PCA transformed components and transaction class.

In [ ]:
plt.figure(figsize=(10, 8))
sub_cols = ["Time", "Amount", "V1", "V2", "V3", "V4", "V10", "V12", "V14", "V17", "Class"]
corr = df[sub_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Correlation Matrix of Key Features and Fraud Class")
plt.show()

## 7. Key Findings & Engineering Implications
1. **Severe Imbalance (~0.17% fraud in ULB):** Accuracy is strictly unusable; PR-AUC (Average Precision) and Sensitivity (Recall) must govern model selection.
2. **Outlier Skew in Amount:** RobustScaler is preferred over standard z-score scaling because median and interquartile range (IQR) resist outlier distortion.
3. **Chronological Splitting:** Random train/test shuffling leads to temporal data leakage in transaction streaming environments; chronological splits benchmark realistic production degradation.
4. **Discriminative Signatures:** Features V14, V17, and V12 strongly indicate fraud when their values plunge into negative territory, whereas V4 surges positively.